# Kaggle Medical Audio Processing Pipeline

This notebook processes medical audio files (Arabic) using:
- **WhisperX large-v3** for transcription
- **MMed-Llama-3-8B** for correction and SOAP notes
- **GPU T4** for fast processing

## Steps:
1. Install dependencies (Cell 2) - ~3 minutes
2. Run pipeline (Cell 3) - processes all audio files
3. Download results from `/kaggle/working`

In [ ]:
# ============================================================================
# CELL 1: Install Dependencies (Run this FIRST)
# ============================================================================

print("=" * 80)
print("INSTALLING DEPENDENCIES FOR KAGGLE")
print("=" * 80)
print()

# Clean up
print("🧹 Cleaning up...")
!pip uninstall -y transformers whisperx bitsandbytes accelerate -q 2>/dev/null

# Install EXACT compatible versions (critical!)
print("\n📦 Step 1/3: Installing Transformers & Accelerate...")
!pip install -q 'transformers==4.44.0' 'accelerate>=0.27.0'
print("✅ Done")

# Install bitsandbytes
print("\n📦 Step 2/3: Installing BitsAndBytes...")
!pip install -q bitsandbytes
print("✅ Done")

# Install WhisperX (will upgrade numpy to 2.0.x - this is OK)
print("\n📦 Step 3/3: Installing WhisperX...")
!pip install -q git+https://github.com/m-bain/whisperx.git
print("✅ Done")

# Verification
print("\n" + "=" * 80)
print("VERIFYING INSTALLATION")
print("=" * 80)

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

try:
    import transformers
    print(f"✅ Transformers: {transformers.__version__}")
    # Critical check
    if transformers.__version__ != "4.44.0":
        print(f"⚠️  WARNING: Expected 4.44.0, got {transformers.__version__}")
        print("   Restart kernel (Kernel → Restart) then re-run this cell")
except Exception as e:
    print(f"❌ Transformers: {e}")

try:
    import whisperx
    print(f"✅ WhisperX: Installed")
    # Test import that was failing
    from transformers import Pipeline
    print(f"✅ Transformers.Pipeline: Working")
except Exception as e:
    print(f"❌ WhisperX: {e}")
    print("   SOLUTION: Restart kernel (Kernel → Restart) then re-run this cell")

try:
    import bitsandbytes as bnb
    print(f"✅ BitsAndBytes: Installed")
except Exception as e:
    print(f"⚠️  BitsAndBytes: {e}")

import numpy as np
import scipy
print(f"✅ NumPy: {np.__version__}")
print(f"✅ SciPy: {scipy.__version__}")

print("\n" + "=" * 80)
print("⚠️  IMPORTANT: After running this cell:")
print("   1. Click 'Kernel' → 'Restart' (or Ctrl+M+.)") 
print("   2. Re-run this cell to verify")
print("   3. Then run Cell 3 (pipeline)")
print("=" * 80)


In [ ]:
# ============================================================================
# CELL 2: Main Processing Pipeline
# ============================================================================

import os
import sys
import time
import json
import torch
import whisperx
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pathlib import Path

# ============================================================================
# SETUP
# ============================================================================

print("=" * 80)
print("KAGGLE AUDIO PROCESSING PIPELINE")
print("=" * 80)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU - will use CPU (very slow)")

print(f"Device: {DEVICE}\n")

# Paths
INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"
MODEL_CACHE = "/kaggle/working/models"

os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['TRANSFORMERS_CACHE'] = MODEL_CACHE
os.environ['HF_HOME'] = MODEL_CACHE

print(f"📁 Input: {INPUT_DIR}")
print(f"📁 Output: {WORKING_DIR}")
print(f"📁 Cache: {MODEL_CACHE}\n")

# ============================================================================
# LOAD MODELS
# ============================================================================

def load_asr_model():
    """Load WhisperX"""
    print("📥 Loading WhisperX large-v3...")
    print("   First run: ~3GB download, 2-5 mins")
    print("   Cached: ~30s")
    start = time.time()
    
    model = whisperx.load_model(
        "large-v3",
        device=DEVICE,
        compute_type=COMPUTE_TYPE,
        language="ar",
        download_root=MODEL_CACHE
    )
    
    print(f"✅ Loaded in {time.time()-start:.1f}s\n")
    return model

def load_llm_model():
    """Load Medical LLM"""
    print("📥 Loading MMed-Llama-3-8B...")
    
    if DEVICE == "cuda":
        print("   Using 4-bit quantization")
        print("   First run: ~8GB download, 5-10 mins")
        print("   Cached: ~3-5 mins")
    else:
        print("   Using 8-bit quantization (CPU)")
        print("   This will take 13-20 mins")
    
    start = time.time()
    model_name = "Henrychur/MMed-Llama-3-8B"
    
    if DEVICE == "cuda":
        config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
    else:
        config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True
        )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODEL_CACHE)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=config,
        device_map="auto",
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
        cache_dir=MODEL_CACHE
    )
    
    elapsed = time.time() - start
    print(f"✅ Loaded in {elapsed:.1f}s ({elapsed/60:.1f} mins)\n")
    return model, tokenizer

# ============================================================================
# PROCESS AUDIO
# ============================================================================

def transcribe_audio(audio_path, asr_model, dialect="egypt"):
    """ASR Transcription"""
    print("=" * 80)
    print("ASR TRANSCRIPTION")
    print("=" * 80)
    print(f"File: {audio_path}")
    print(f"Dialect: {dialect}\n")
    
    start = time.time()
    
    # Load and transcribe
    print("📂 Loading audio...")
    audio = whisperx.load_audio(audio_path)
    duration = len(audio) / 16000
    print(f"   Duration: {duration:.1f}s")
    
    print("🎤 Transcribing...")
    result = asr_model.transcribe(audio, language="ar", batch_size=16)
    
    print("🔍 Aligning...")
    model_a, metadata = whisperx.load_align_model(language_code="ar", device=DEVICE)
    result = whisperx.align(result["segments"], model_a, metadata, audio, DEVICE, return_char_alignments=False)
    
    elapsed = time.time() - start
    print(f"\n✅ Complete in {elapsed:.1f}s ({elapsed/duration:.2f}x RT)")
    print(f"   Segments: {len(result['segments'])}")
    
    full_text = " ".join([seg["text"] for seg in result["segments"]])
    print(f"   Text: {full_text[:100]}...\n")
    
    return result, full_text

def correct_transcription(text, llm_model, tokenizer):
    """LLM Correction"""
    print("=" * 80)
    print("LLM CORRECTION")
    print("=" * 80)
    print(f"Input: {text[:100]}... ({len(text)} chars)\n")
    
    prompt = f"""صحح الأخطاء في هذا النص الطبي: {text}

النص المصحح:"""
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}
    
    print(f"🤖 Generating... (GPU: ~5-10s, CPU: ~20-30 mins)")
    start = time.time()
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
            repetition_penalty=1.1
        )
    
    print(f"✅ Complete in {time.time()-start:.1f}s")
    
    corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "النص المصحح:" in corrected:
        corrected = corrected.split("النص المصحح:")[-1].strip()
    corrected = corrected.replace(prompt, "").strip()
    
    if len(corrected) > len(text) * 3 or len(corrected) < 5:
        print("⚠️  Malformed output, using original")
        corrected = text
    
    print(f"   Output: {corrected[:100]}...\n")
    return corrected

def generate_soap_note(text, llm_model, tokenizer):
    """Generate SOAP Note"""
    print("=" * 80)
    print("SOAP NOTE GENERATION")
    print("=" * 80)
    
    prompt = f"""قم بتحويل هذه المحادثة الطبية إلى تقرير SOAP:

المحادثة: {text}

التقرير (S.O.A.P):"""
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}
    
    print(f"🤖 Generating... (GPU: ~10-20s, CPU: ~40-60 mins)")
    start = time.time()
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )
    
    print(f"✅ Complete in {time.time()-start:.1f}s")
    
    soap = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "التقرير" in soap:
        soap = soap.split("التقرير")[-1].strip()
    soap = soap.replace(prompt, "").strip()
    
    print(f"   Output: {soap[:100]}...\n")
    return soap

# ============================================================================
# MAIN
# ============================================================================

def main():
    # Find audio files
    print("🔍 Finding audio files...")
    extensions = ['.mp3', '.wav', '.m4a', '.flac', '.ogg']
    audio_files = []
    for root, dirs, files in os.walk(INPUT_DIR):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                audio_files.append(os.path.join(root, file))
    
    if not audio_files:
        print(f"❌ No audio found in {INPUT_DIR}")
        return
    
    print(f"✅ Found {len(audio_files)} file(s):")
    for f in audio_files:
        print(f"   - {f}")
    print()
    
    # Load models
    print("=" * 80)
    print("LOADING MODELS")
    print("=" * 80)
    asr_model = load_asr_model()
    llm_model, llm_tokenizer = load_llm_model()
    
    # Process files
    results = []
    for i, audio_path in enumerate(audio_files, 1):
        print(f"\n{'='*80}")
        print(f"PROCESSING FILE {i}/{len(audio_files)}")
        print(f"{'='*80}\n")
        
        try:
            asr_result, full_text = transcribe_audio(audio_path, asr_model)
            corrected = correct_transcription(full_text, llm_model, llm_tokenizer)
            soap = generate_soap_note(corrected, llm_model, llm_tokenizer)
            
            # Save result
            output_file = os.path.join(WORKING_DIR, f"{Path(audio_path).stem}_result.json")
            result = {
                "audio_file": audio_path,
                "device": DEVICE,
                "asr_result": {"segments": asr_result["segments"], "full_text": full_text},
                "corrected_text": corrected,
                "soap_note": soap,
                "status": "success"
            }
            
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            
            print(f"✅ Saved: {output_file}\n")
            results.append(result)
            
        except Exception as e:
            print(f"❌ Error: {e}\n")
            results.append({"audio_file": audio_path, "status": "error", "error": str(e)})
    
    # Save summary
    summary_file = os.path.join(WORKING_DIR, "summary.json")
    summary = {
        "total_files": len(audio_files),
        "successful": len([r for r in results if r.get("status") == "success"]),
        "failed": len([r for r in results if r.get("status") == "error"]),
        "device": DEVICE,
        "results": results
    }
    
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    
    print("=" * 80)
    print("COMPLETE")
    print("=" * 80)
    print(f"✅ Success: {summary['successful']}/{summary['total_files']}")
    print(f"❌ Failed: {summary['failed']}/{summary['total_files']}")
    print(f"\n📁 Results in /kaggle/working/")
    print("   Download from Output tab")
    print("=" * 80)

# Run pipeline
main()